# 07 — Data Management

This notebook shows how to work with the `tutorial/data/` folder:

1. Generate sample HDF5 files (if not already done)
2. List and browse data files
3. Load an `ExperimentData` from HDF5
4. Compare results across runs
5. Load and inspect a `CalibrationStore`

**Pre-requisite:** run the data generator once:
```bash
cd SQC_soc
python tutorial/data/generate.py
```

In [ ]:
import sys, os
sys.path.insert(0, '../')

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'tutorial', 'data')
# If running from inside tutorial/
if not os.path.isdir(DATA_DIR):
    DATA_DIR = 'data'

print('Data directory:', DATA_DIR)
print('Files:', [f for f in os.listdir(DATA_DIR) if not f.startswith('.')])

## Step 0 — Generate the sample files (only needed once)

In [ ]:
h5_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.h5')]

if not h5_files:
    print('No HDF5 files found — running generator...')
    import subprocess
    result = subprocess.run(
        [sys.executable, os.path.join(DATA_DIR, 'generate.py')],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
    h5_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.h5')]

print('HDF5 files found:', h5_files)

## 1. List all data files

In [ ]:
from reconstruct.data.manager import list_data_files

entries = list_data_files(DATA_DIR)

print(f'{'Filename':<30} {'Experiment':<25} {'Timestamp'}')
print('-' * 75)
for e in entries:
    name = os.path.basename(e['path'])
    print(f"{name:<30} {e['experiment']:<25} {e['timestamp']}")

## 2. Load a single ExperimentData from HDF5

In [ ]:
from reconstruct import ExperimentData

t1_path = os.path.join(DATA_DIR, 't1.h5')
t1 = ExperimentData.load(t1_path)

print('Experiment type :', t1.experiment_type)
print('ID              :', t1.experiment_id)
print('Timestamp       :', t1.timestamp)
print('Quality         :', t1.quality)
print('scalar_result   :', t1.scalar_result)
print('fit_result      :', t1.fit_result)
print('x_axis shape    :', t1.x_axis.shape if t1.x_axis is not None else None)
print('Config snapshot :', {k: t1.config[k] for k in list(t1.config)[:5]})

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t1.x_axis, t1.y_axis, 'o', ms=3, label='data')

if t1.fit_params is not None and len(t1.fit_params) >= 3:
    A, T1_fit, C = t1.fit_params[:3]
    t_fit = np.linspace(t1.x_axis[0], t1.x_axis[-1], 400)
    ax.plot(t_fit, A * np.exp(-t_fit / T1_fit) + C, 'r-',
            lw=1.5, label=f'T1 = {T1_fit:.1f} µs')

ax.set_xlabel('Wait time (µs)')
ax.set_ylabel('Signal (a.u.)')
ax.set_title('T1 — loaded from HDF5')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Load all experiments and compare scalar results

In [ ]:
loaded = {}
for e in entries:
    try:
        data = ExperimentData.load(e['path'])
        key  = os.path.splitext(os.path.basename(e['path']))[0]
        loaded[key] = data
    except Exception as exc:
        print(f'  Could not load {e["path"]}: {exc}')

print(f"{'Name':<20} {'Quality':<12} {'Scalar result'}")
print('-' * 50)
for name, d in loaded.items():
    val = f'{d.scalar_result:.4f}' if d.scalar_result is not None else 'N/A'
    print(f'{name:<20} {d.quality.value:<12} {val}')

## 4. Inspect the CalibrationStore

In [ ]:
from reconstruct import CalibrationStore

# Try the pre-generated store first, fall back to the hand-crafted example
generated_path = os.path.join(DATA_DIR, 'cal_store_generated.json')
example_path   = os.path.join(DATA_DIR, 'cal_store_Q1.json')

store_path = generated_path if os.path.exists(generated_path) else example_path
store = CalibrationStore(store_path)

print(store)
print()
print(store.summary())

In [ ]:
# Check which parameters are stale
FRESHNESS = {
    'res_freq_ge': 48,
    'qb_freq_ge':  12,
    'pi_gain_ge':  12,
    'T1_us':       24,
    'T2r_us':      24,
}

for qubit in store.all_qubits():
    print(f'\n[{qubit}]')
    for param, max_h in FRESHNESS.items():
        val   = store.get(qubit, param)
        stale = store.is_stale(qubit, param, max_age_hours=max_h)
        tag   = '*** STALE ***' if stale else 'fresh'
        val_s = f'{val:.4f}' if isinstance(val, float) else str(val)
        print(f'  {param:<20} = {val_s:<10}  [{tag}]')

## 5. Export to JSON for logging / dashboards

In [ ]:
import json

# ExperimentData → JSON-safe dict
t1_dict = t1.to_dict()
print(json.dumps(
    {k: t1_dict[k] for k in ('experiment_type', 'timestamp', 'scalar_result', 'quality')},
    indent=2, default=str
))

In [ ]:
# CalibrationStore → flat dict (ready to feed back into ExperimentConfig)
for qubit in store.all_qubits():
    flat = store.to_flat_dict(qubit)
    print(f'{qubit}: {json.dumps(flat, indent=2)}')

## Data folder layout

```
tutorial/data/
├── generate.py              ← run once to create sample .h5 files
├── system_cfg_example.py    ← hardware config template for your lab
├── cal_store_Q1.json        ← hand-crafted example CalibrationStore
├── cal_store_generated.json ← written by generate.py
├── res_spec.h5              \ 
├── qubit_spec.h5             |
├── power_rabi.h5             |  written by generate.py
├── t1.h5                     |
├── ramsey.h5                 |
└── spin_echo.h5             /
```

In a real lab session you would point `DATA_PATH` in `system_cfg.py` to your
data drive (e.g. `D:/data/`).  The `list_data_files()` function recursively
scans any directory for `.h5` files.